# DentAware: run the full study on Colab

Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Results are saved to Google Drive, so if Colab disconnects you can reconnect, rerun cells 1–4, and continue from the stage you were on (finished detection models are reused automatically).

In [ ]:
# 1. GPU check
!nvidia-smi -L

In [ ]:
# 2. Save everything to Google Drive
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/dentaware_runs/main'

**3. Get the code.** Either upload `dentaware.zip` with the file browser on the left and unzip it, or clone your GitHub copy (edit the URL).

In [ ]:
import os
if os.path.exists('/content/dentaware.zip'):
    !unzip -oq /content/dentaware.zip -d /content
else:
    !git clone https://github.com/vohongvyle/dentaware.git /content/dentaware
%cd /content/dentaware
!pip install -q -r requirements.txt

In [ ]:
# 4. Download the dataset (Kaggle mirror of Zhang et al., 2023)
import kagglehub
DATA = kagglehub.dataset_download('truthisneverlinear/childrens-dental-panoramic-radiographs-dataset')
print(DATA)

## Optional: 5-minute smoke test
Tiny settings, meaningless numbers. Use it once to confirm everything installs and runs.

In [ ]:
!python -m dentaware.pipeline --data-root "$DATA" --out /content/smoke --stage all --quick

Check the web demo works (the smoke-test models are tiny, so the findings will be nonsense). Click the public link it prints. Stop the cell when done.

In [ ]:
from dentaware.demo import build_app
build_app('/content/smoke', DATA).launch(share=True)

## Step 1: audit the data (1 minute)
Check the class counts and the problems it reports before training anything.

In [ ]:
!python -m dentaware.pipeline --data-root "$DATA" --out "$OUT" --stage audit
import pandas as pd
pd.read_csv(f'{OUT}/audit/class_counts.csv')

## Step 2: tooth segmentation, three strategies (about 1–1.5 hours) — RQ3

In [ ]:
!python -m dentaware.pipeline --data-root "$DATA" --out "$OUT" --stage seg
pd.read_csv(f'{OUT}/seg/seg_results.csv')

## Step 3: disease detection, baseline vs. tooth-aware (about 2–3 hours) — RQ1, RQ2
If the session disconnects, rerun cells 1–4 and this cell: finished fold models are reused.

In [ ]:
!python -m dentaware.pipeline --data-root "$DATA" --out "$OUT" --stage det
pd.read_csv(f'{OUT}/det/det_results.csv')[['variant','split','n_images','mAP50','ci_low','ci_high']]

In [ ]:
import json
print(json.dumps(json.load(open(f'{OUT}/det/det_summary.json'))['comparisons'], indent=1))

## Step 4: calibration, referral and tooth assignment — RQ4

In [ ]:
!python -m dentaware.pipeline --data-root "$DATA" --out "$OUT" --stage reliability
print(json.dumps(json.load(open(f'{OUT}/reliability/reliability_summary.json')), indent=1))
pd.read_csv(f'{OUT}/reliability/lesion_teeth_test.csv').head(15)

## Step 5: figures and speed

In [ ]:
!python -m dentaware.pipeline --data-root "$DATA" --out "$OUT" --stage figures
!python -m dentaware.pipeline --data-root "$DATA" --out "$OUT" --stage latency
from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'{OUT}/figures/*.png')) + sorted(glob.glob(f'{OUT}/reliability/*.png')):
    print(f); display(Image(f))

## Next
Copy the numbers from `seg_results.csv`, `det_results.csv` and `reliability_summary.json` into `report/paper_template.md`. Report the confidence intervals, not just the point estimates.

## Step 6: the web demo
Upload a panoramic X-ray or click a test-set example. It outlines the teeth, shows each finding with its tooth and calibrated confidence, and says when a radiograph should go to a dentist. `share=True` prints a public link that works on your phone for about a week while this notebook is running. Stop the cell to shut it down.

In [ ]:
from dentaware.demo import build_app
build_app(OUT, DATA).launch(share=True)